# Study report: mask-instrumentation

The study checks the instrument of the study `mask-time`: the interrupt
mask hooks (M4) of the tree `ws-up-irqoff`. It asks if the hooks measure
the mask time of each call site, and if they leave the image unchanged
when they are off (`CONFIG_ARM_IRQ_MASK_HOOKS=n`). When the hooks are on,
the latency histograms are not data: the hooks make each mask section
longer, and they add time to the wake path (`LAT META m4=1`). The
analysis uses the M4 site tables; the IRQ tail of the same run is only a
cross check.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. LATBENCH_INSTANCE: the instance folder ("study.py analysis notebooks" sets it). Else
# LATBENCH_RESULTS: the results root of a lab space (for example <lab>/results/kernel-latency), and the
# notebook finds its instance in it. LATBENCH_REPO: the safety repo; else the repo of the current folder.
import os
import subprocess
import sys
from pathlib import Path

INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or Path(
    os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-measurement-validity@nucleo_g474re/st-mask-instrumentation")
REPO = Path(os.environ.get("LATBENCH_REPO") or subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True, check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

In [ ]:
import plotly.graph_objects as go
from studylib import stats  # noqa: E402

## H1: the image with the hooks off (wake-m4-image-check)

The experiment builds two points of `wake-timers-1m` on `ws-up` and on
`ws-up-irqoff`, with the hooks off. H1 expects the same image for each
point; only the version strings (`BUILD_VERSION`, the `git describe` of
each Zephyr tree) can differ. `image_sha256` is the digest of the loaded
segments of the ELF. `code_sha256` is the same digest with the
`bench_rev` string set to zeros. The version strings thus count in both
digests, so a person gives the verdict of H1 a.

In [ ]:
if nbfig.built(inst, "wake-m4-image-check"):
    img = inst.rows("wake-m4-image-check").drop_duplicates(["tree", "point"]).merge(
        inst.meta("wake-m4-image-check")[["tree", "point", "zephyr"]], on=["tree", "point"])
    display(img[["point", "tree", "zephyr", "image_sha256", "code_sha256"]].sort_values(["point", "tree"]))

The campaign also ran the two images of each point on the board. The
table compares their latency histograms bin for bin.

In [ ]:
if nbfig.built(inst, "wake-m4-image-check"):
    reps = {(t, p): rep for _, t, p, rep in inst.points("wake-m4-image-check")}
    same = []
    for p in sorted({p for _, p in reps}):
        a, b = reps.get(("ws-up", p)), reps.get(("ws-up-irqoff", p))
        if a and b and a.get("hists") and b.get("hists"):
            for m in sorted(set(a["hists"]) & set(b["hists"])):
                same.append({"point": p, "metric": m, "n": a["hists"][m]["n"], "max (ws-up)": a["hists"][m]["max"],
                             "max (ws-up-irqoff)": b["hists"][m]["max"],
                             "the same, bin for bin": stats.hist_identical(a["hists"][m], b["hists"][m])})
    display(pd.DataFrame(same) if same else "no latency histograms in the runs of wake-m4-image-check")

## The M4 site table (wake-m4-validate)

A site is the return address of the start hook (it points into the code
that masks), the exception number at the start (`t` in a thread, `e14`
PendSV, `e15` SysTick, `e22` EXTI0) and the kind (`mask`: a mask
section, `isr`: an ISR body). `max_net` is the max minus the floor of
the calibration of the same log. The first table gives the longest site
of each kind for each point. The second table gives all sites of the
point `cal-t0` (no load).

In [ ]:
st = inst.sites("wake-m4-validate") if nbfig.built(inst, "wake-m4-validate") else None
if st is not None and len(st):
    top = st.sort_values("max_net", ascending=False).groupby(["point", "kind"], sort=False).head(1)
    order = list(dict.fromkeys(st.point))
    display(top.assign(o=top.point.map(order.index)).sort_values(["o", "kind"]).drop(columns=["o", "tree"]))
    display(st[st.point == "cal-t0"].sort_values(["kind", "max_net"], ascending=[True, False]).drop(columns=["tree"]))

## H2 a, b: the calibration of each run

`irqmask_start()` measures the hooks before the run, as the minimum of
256 tries. `m4_floor_cyc` is the value of an empty section
(`irq_unlock(irq_lock())`); each section value contains it.
`m4_pair_cost_cyc` is the extra CPU time of the hooks for one section,
against the same BASEPRI writes without hooks. The bands are the limits
of H2 a and H2 b, from an estimate from the disassembly of the hooks.

In [ ]:
if nbfig.built(inst, "wake-m4-validate"):
    cal = inst.meta("wake-m4-validate")[["point", "m4_floor_cyc", "m4_pair_cost_cyc", "m4_isr_floor_cyc",
                                         "m4_unmatched_end", "m4_begin_open"]]
    fig = go.Figure()
    for k, (col, lo, hi, cid) in enumerate([("m4_floor_cyc", 15, 30, "H2 a"), ("m4_pair_cost_cyc", 60, 150, "H2 b")]):
        fig.add_hrect(y0=lo, y1=hi, fillcolor=nbfig.color(k), opacity=0.12, line_width=0,
                      annotation_text=f"limit of {cid}", annotation_position="top left")
        fig.add_trace(go.Scatter(x=cal.point, y=cal[col], mode="markers+lines", name=col,
                                 line={"color": nbfig.color(k), "width": 2}, marker={"size": 10, "symbol": nbfig.SYMBOLS[k]},
                                 hovertemplate="%{x}: %{y} cycles<extra>" + col + "</extra>"))
    nbfig.style(fig, "Calibration of each run, against the limits of H2 a and H2 b", "point", "cycles").show()
    display(cal)

## H2 c, d, e: the cost of the hooks on the wake path

The wake from WFI runs a begin and an end hook before `cpsie i`, and the
ISR enter hook runs before the ISR. H2 c and d compare the p50 of the
point `cal-t0` with the point `t0-default` of `wake-timers-1m` on
`ws-up` (an import, no hooks). H2 e reads the wake window of
`arch_cpu_idle`: the section starts after the wake-up and ends at
`cpsie i`.

In [ ]:
rows = []
if nbfig.built(inst, "wake-m4-validate"):
    cal_rep = next(rep for _, _, p, rep in inst.points("wake-m4-validate") if p == "cal-t0")
    if nbfig.built(inst, "wake-timers-1m"):
        ref = next((rep for _, t, p, rep in inst.points("wake-timers-1m") if t == "ws-up" and p == "t0-default"), None)
        for cid, m, lo, hi in (("H2 c", "irq", 60, 150), ("H2 d", "isr2thr", 200, 400)):
            if ref is not None:
                a, b = stats.value(cal_rep, m, "p50"), stats.value(ref, m, "p50")
                rows.append({"check": f"{cid}: {m} p50 - wake-timers-1m t0-default", "series": "cal-t0", "value": a - b,
                             "cal-t0": a, "reference": b, "lo": lo, "hi": hi})
    v = stats.value(cal_rep, "m4.mask[arch_cpu_idle]", "max(max_net)")
    rows.append({"check": "H2 e: arch_cpu_idle max_net", "series": "cal-t0", "value": v, "cal-t0": v, "reference": None,
                 "lo": 0, "hi": 19})
wake = pd.DataFrame(rows)
if len(wake):
    nbfig.limits(wake, "The cost of the hooks on the wake path (point cal-t0)", "cycles").show()
wake

## H3: the probe

The probe thread masks the interrupts for exactly 5000 DWT cycles every
10 ms (point `probe-t0`). Its site must show this length plus the floor,
so H3 a reads `min_net` and `max_net` of the site `probe_fn`. An edge
that falls into a probe section waits for the end of the section and the
end hook, so the IRQ tail of the same run must show the probe too (H3
b).

In [ ]:
if nbfig.built(inst, "wake-m4-validate"):
    pr = {p: rep for _, _, p, rep in inst.points("wake-m4-validate")}
    if "probe-t0" in pr:
        rep = pr["probe-t0"]
        probe = pd.DataFrame([
            {"check": "H3 a: probe_fn min_net", "series": "probe-t0", "value": stats.value(rep, "m4.mask[probe_fn]", "min_net"),
             "lo": 5000, "hi": 5010},
            {"check": "H3 a: probe_fn max_net", "series": "probe-t0", "value": stats.value(rep, "m4.mask[probe_fn]", "max_net"),
             "lo": 5000, "hi": 5010},
            {"check": "H3 b: irq max", "series": "probe-t0", "value": stats.value(rep, "irq", "max"), "lo": 5000, "hi": 5400}])
        fig = nbfig.limits(probe, "The probe of 5000 cycles: site and IRQ tail", "cycles")
        fig.update_yaxes(range=[4900, 5500])
        fig.show()
        display(probe)
        fig = nbfig.ccdf({p: pr[p]["hists"] for p in ("cal-t0", "probe-t0") if p in pr},
                         "IRQ entry with and without the probe (the histograms of an M4 run are a cross check only)",
                         ["irq"])
        fig.add_vline(x=5000, line_dash="dash", line_color="#52514c", annotation_text="probe length: 5000 cycles")
        fig.show()

## H4: the IRQ max against the longest mask section

With SysTick one level lower (`LATBENCH_SYSTICK_PRIO=1`, points
`*-lower`), the SysTick ISR body does not block the EXTI0 ISR, and the
longest mask section sets the IRQ max. The rest is the IRQ entry plus
the tail of the end hook. H4 a expects the IRQ max minus the longest
mask section in 100..300 cycles. If the IRQ max is clearly larger, M4
does not see a mask path. The first figure shows the check; the second
shows the two values for all points (the colour is the value, the
x axis the point).

In [ ]:
if nbfig.built(inst, "wake-m4-validate"):
    tail = []
    for _, _, p, rep in inst.points("wake-m4-validate"):
        top = stats.value(rep, "m4.mask", "max(max_net)")
        tail.append({"point": p, "irq max": stats.value(rep, "irq", "max"), "longest mask section (max_net)": top,
                     "function": stats.value(rep, "m4.mask", "top_place"),
                     "irq max - longest mask section": stats.value(rep, "irq", "max") - top})
    tail = pd.DataFrame(tail)
    chk = tail[tail.point.isin(["t64-lower", "t256-lower"])]
    nbfig.limits(pd.DataFrame({"check": "H4 a: " + chk.point, "series": "irq max - longest mask section",
                         "value": chk["irq max - longest mask section"], "lo": 100, "hi": 300}),
           "H4 a: the rest of the IRQ max over the longest mask section", "cycles").update_layout(height=400).update_yaxes(
        range=[0, max([350] + [v * 1.1 for v in chk["irq max - longest mask section"]])]).show()
    fig = go.Figure()
    for k, col in enumerate(["irq max", "longest mask section (max_net)"]):
        fig.add_trace(go.Scatter(x=tail.point, y=tail[col], mode="markers", name=col, customdata=tail.function,
                                 marker={"color": nbfig.color(k), "size": 12, "symbol": nbfig.SYMBOLS[k]},
                                 hovertemplate="%{x}: %{y} cycles<br>%{customdata}<extra>" + col + "</extra>"))
    nbfig.style(fig, "IRQ max and the longest mask section of each point", "point", "cycles", yaxis_type="log").show()
    display(tail)